<div style="border-top:4px solid #0f766e;padding:16px">DATA WAREHOUSING WITH APACHE DORIS · OPTIONAL LAB</div>

# Optional Lab 5A: Kafka → Routine Load → Doris

**Module 5: Batch and Continuous Data Ingestion** · [Lesson](course.md) · [Quiz](quiz05_load_methods_and_retry_safety.ipynb) · [Reading: Continuous Ingestion](optional05_continuous_ingestion.md)

A Kafka topic receives order events one at a time. A Routine Load job in Doris consumes them continuously, writes them in small batches, and keeps its own position in the topic. You send orders, read the job's progress, pause the job to build a backlog, resume it, send a duplicate and an update, and see what a bad message does to the job.

Plan about 30 minutes. The Lab is optional: no other Lab depends on it.

**What you need**

- Docker with at least 8 GiB of memory and 4 CPUs (Docker Desktop: Settings → Resources). In this Lab Doris used at most 2.9 GiB and Kafka 0.4 GiB, measured on Docker Desktop with Apple silicon. The first cell checks Docker's capacity before it starts anything.
- About 640 MB to download the first time (`apache/kafka:3.9.0`).
- The course sandbox from Lab 1. The first cell recreates the Doris container with a 6 GiB memory limit; its data stays. Finish the other notebooks first. Lab 1's **Start and Connect** cell removes the limit again: run it before you go back to the main Labs, whose BE needs more than the 2.4 GiB that the limit leaves it.

The Lab rebuilds only `ext_kafka_orders` and, in the practice, `ext_kafka_practice`, both in the database `dw_course_l1_streaming`, so the main course's tables do not change. Every run uses a new topic and a new job. Its services are the Compose project `doris-warehousing-streaming`: the end of the Lab says how to stop them, and the course README's [Clean up](../../README.md#clean-up) section how to remove them.

If you cannot run the Lab, read it anyway: each step's **Expected result** shows what a run prints.

In [ ]:
from pathlib import Path
import os, sys
from uuid import uuid4
course = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "dw_course").is_dir())
sys.path.insert(0, str(course))
from dw_course.docker_runtime import prepare_environment, connect_sandbox
from dw_course.runtime import expect
from dw_course.streaming import (
    prepare_streaming, kafka, produce, wait_rows, check_routine_load, cleanup_kafka_routine_load,
)
prepare_environment(start=True, streaming=True, streaming_profile="kafka")
os.environ["DW_DATABASE"] = "dw_course_l1_streaming"
lab = connect_sandbox()
print("Isolated lab database:", lab.database)

prepare_streaming("kafka", start=True)

**Expected result**

`Docker capacity: … GiB RAM, … CPUs`, at least 8 GiB and 4 CPUs, then the startup panel with six green steps. The first time, Compose recreates the Doris container with the 6 GiB limit, which takes about 20 seconds. Then `Isolated lab database: dw_course_l1_streaming`, and Compose starts `doris-warehousing-streaming-kafka-1` after downloading the Kafka image on a first run.

With less memory, the cell stops before starting anything and says how much the Lab needs.

## 1. Create the target table, a topic, and the load job

The setup first stops stale jobs that an earlier run of this Lab left behind, after checking their names and target tables; it refuses to touch any other Routine Load job. Then it rebuilds `ext_kafka_orders`, a Unique Key table that keeps one row per order, creates a topic with one partition, and creates the job:

| Job setting | Meaning |
| --- | --- |
| `"format"="json"` | Each Kafka message is one JSON object; its keys are the column names |
| `"max_batch_interval"="5"` | Each load task runs for at most 5 seconds before it commits what it read |
| `"max_error_number"="0"`, `"strict_mode"="true"` | Pause the job at the first row that does not fit the table |
| `"property.kafka_default_offsets"="OFFSET_BEGINNING"` | A new job starts at the beginning of each partition |

Doris keeps the job's position, its **progress**, itself, with each committed task; it is not the committed offset of an ordinary Kafka consumer group.

In [ ]:
cleanup_kafka_routine_load(lab)
expect(lab.query("SHOW ROUTINE LOAD"), [], title="No earlier Routine Load job is still active")
lab.execute("DROP TABLE IF EXISTS ext_kafka_orders")
lab.execute("""CREATE TABLE ext_kafka_orders (
    order_id BIGINT NOT NULL, customer_id BIGINT NOT NULL,
    order_status VARCHAR(32) NOT NULL, order_amount DECIMAL(18,2) NOT NULL
) UNIQUE KEY(order_id) DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES("replication_num"="1", "enable_unique_key_merge_on_write"="true")""")
run_id = uuid4().hex[:12]
topic = "course_orders_" + run_id
job_name = "course_orders_" + run_id
print(kafka("kafka-topics.sh", "--create", "--topic", topic, "--partitions", "1", "--replication-factor", "1"))
load_sql = f'''CREATE ROUTINE LOAD {job_name} ON ext_kafka_orders
COLUMNS(order_id, customer_id, order_status, order_amount)
PROPERTIES("format"="json", "desired_concurrent_number"="1",
           "max_batch_interval"="5", "max_error_number"="0", "strict_mode"="true")
FROM KAFKA("kafka_broker_list"="course-stream-kafka:9092", "kafka_topic"="{topic}",
           "property.kafka_default_offsets"="OFFSET_BEGINNING")'''
print(load_sql)
lab.execute(load_sql);

**Expected result**

A green check that no earlier Routine Load job is still active, then `Created topic course_orders_<12 hexadecimal digits>.`, with a warning from Kafka about periods and underscores in topic names that does not matter here, then the `CREATE ROUTINE LOAD` statement. The job and the topic share the name. No error means that the job exists; it waits for messages.

## 2. Send two orders and wait until they are visible

The producer runs inside the Kafka container and writes two JSON messages. A message in Kafka is not yet a row in Doris: the next cell waits until both rows can be queried, at most 3 minutes, and checks the job on every attempt. If the job pauses or stops, the wait stops with the job's reason and its error URL.

In [ ]:
rows = [dict(order_id=910001, customer_id=1, order_status="CREATED", order_amount="100.00"),
        dict(order_id=910002, customer_id=2, order_status="CREATED", order_amount="200.00")]
produce(topic, rows)
baseline = [[910001,1,"CREATED","100.00"],[910002,2,"CREATED","200.00"]]
expect(wait_rows(lab,"ext_kafka_orders",baseline, check_health=lambda: check_routine_load(lab, job_name), description="initial Kafka rows"), baseline,
       title="Orders 910001 and 910002 are visible")
lab.sql(f"SHOW ROUTINE LOAD FOR {job_name}", title="Consumption state and progress",
        columns=["Name", "State", "Progress", "Lag", "Statistic", "ReasonOfStateChanged"]);

**Expected result**

The wait card turns green within about 10 seconds, and a green check confirms the two rows 910001 and 910002. The job row shows `State` RUNNING and `Progress` `{"0":"1"}`: in partition 0, the job has loaded up to offset 1, the second message, since offsets count from 0. `Lag` `{"0":0}` means that no message waits, and `Statistic` shows `"loadedRows":2` and `"errorRows":0`.

## 3. Pause the job and build a backlog

Pause the job, check that it is paused, then send a third order. After 12 seconds, more than two batch intervals, the table must still hold only the first two rows. Pausing keeps the job and its progress, and it does not remove rows that it already loaded.

In [ ]:
import time
lab.execute(f"PAUSE ROUTINE LOAD FOR {job_name}")
paused = lab.sql(f"SHOW ROUTINE LOAD FOR {job_name}", title="The paused job",
                 columns=["Name", "State", "PauseTime", "Progress", "Lag"])
expect(paused.iloc[0]["State"], "PAUSED", title="The job is paused")
produce(topic,[dict(order_id=910003,customer_id=3,order_status="CREATED",order_amount="50.00")])
time.sleep(12)
expect(lab.query("SELECT * FROM ext_kafka_orders ORDER BY order_id"), baseline,
       title="The third order waits in Kafka")

**Expected result**

`State` PAUSED, with a `PauseTime`. Twelve seconds later the table still holds 910001 and 910002: the third message waits in Kafka, after the job's progress.

## 4. Resume, then send a duplicate and an update

Resume the job: it continues from its progress and loads the waiting order. Then send order 910001 again unchanged, and once more with the status PAID. The table keeps one row per order, so the duplicate changes nothing and the update replaces the order's row. A Unique Key table keeps the latest state of each order, not its history; Module 7 resolves events that arrive out of order with versions.

In [ ]:
lab.execute(f"RESUME ROUTINE LOAD FOR {job_name}")
expected = baseline + [[910003,3,"CREATED","50.00"]]
expect(wait_rows(lab,"ext_kafka_orders",expected, check_health=lambda: check_routine_load(lab, job_name), description="new row after resume"),expected)
produce(topic,[rows[0],dict(order_id=910001,customer_id=1,order_status="PAID",order_amount="100.00")])
expected[0][2] = "PAID"
expect(wait_rows(lab,"ext_kafka_orders",expected, check_health=lambda: check_routine_load(lab, job_name), description="updated order after resume"),expected)
lab.sql(f"SHOW ROUTINE LOAD FOR {job_name}", title="Progress after resuming",
        columns=["Name", "State", "Progress", "Lag", "Statistic", "ReasonOfStateChanged"]);
expect(lab.query("SELECT COUNT(*),SUM(order_amount) FROM ext_kafka_orders"), [[3,"350.00"]],
       title="Three orders, 350.00")

**Expected result**

Two green waits: 910003 arrives after `RESUME`, then 910001 becomes PAID. The job row shows `State` RUNNING, `Progress` `{"0":"4"}`, five messages with offsets 0–4, `Lag` `{"0":0}`, and `"loadedRows":5` in `Statistic`. The table has 3 rows totaling 350.00: five messages, but the duplicate and the update replaced the row of 910001, so the row count does not show how many messages arrived.

## 5. Practice: a message that the table rejects

Create a second topic and job, `course_practice_<12 hexadecimal digits>`, with the same settings, loading into a new table `ext_kafka_practice` with the columns of `ext_kafka_orders`. Send a valid order 930001 and an order 930002 whose amount is `not-a-number`.

Before you run it, predict: does the job load 930001? What state does the job end in? Then read the job's `State`, `ReasonOfStateChanged`, `Statistic`, and `ErrorLogUrls`, open the error log, and finally stop the job and delete the topic, so that the next run of this Lab finds no job of yours. The error URL names the BE's address inside the container, `127.0.0.1:8040`; from the notebook, read the same path from `os.environ["DW_BE_HTTP_URL"]`.

In [ ]:
# Your code here

The next cell is a reference solution; it is collapsed. Expand it to compare, or run it.

In [ ]:
import requests
from dw_course.streaming import routine_load_state, wait_for

practice_job = "course_practice_" + uuid4().hex[:12]
lab.execute("DROP TABLE IF EXISTS ext_kafka_practice")
lab.execute("""CREATE TABLE ext_kafka_practice (
    order_id BIGINT NOT NULL, customer_id BIGINT NOT NULL,
    order_status VARCHAR(32) NOT NULL, order_amount DECIMAL(18,2) NOT NULL
) UNIQUE KEY(order_id) DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES("replication_num"="1", "enable_unique_key_merge_on_write"="true")""")
print(kafka("kafka-topics.sh", "--create", "--topic", practice_job, "--partitions", "1", "--replication-factor", "1"))
lab.execute(f'''CREATE ROUTINE LOAD {practice_job} ON ext_kafka_practice
COLUMNS(order_id, customer_id, order_status, order_amount)
PROPERTIES("format"="json", "desired_concurrent_number"="1",
           "max_batch_interval"="5", "max_error_number"="0", "strict_mode"="true")
FROM KAFKA("kafka_broker_list"="course-stream-kafka:9092", "kafka_topic"="{practice_job}",
           "property.kafka_default_offsets"="OFFSET_BEGINNING")''')
produce(practice_job, [dict(order_id=930001, customer_id=1, order_status="CREATED", order_amount="10.00"),
                       dict(order_id=930002, customer_id=2, order_status="CREATED", order_amount="not-a-number")])
wait_for(lambda: routine_load_state(lab, practice_job)["State"], lambda state: state == "PAUSED",
         description="the practice job pauses at the bad message", timeout=120)
lab.sql(f"SHOW ROUTINE LOAD FOR {practice_job}", title="The practice job after the bad message",
        columns=["Name", "State", "Progress", "Statistic", "ReasonOfStateChanged", "ErrorLogUrls"]);
error_url = routine_load_state(lab, practice_job)["ErrorLogUrls"]
print(requests.get(os.environ["DW_BE_HTTP_URL"] + error_url[error_url.index("/api/"):], timeout=10).text)
lab.sql("SELECT * FROM ext_kafka_practice ORDER BY order_id", title="What the practice job loaded");
expect(lab.query("SELECT order_id FROM ext_kafka_practice"), [(930001,)],
       title="The valid order of the batch is loaded")
lab.execute(f"STOP ROUTINE LOAD FOR {practice_job}")
print(kafka("kafka-topics.sh", "--delete", "--topic", practice_job))

**Expected result**

Within about 5 seconds the job is PAUSED. `ReasonOfStateChanged` says `current error rows is more than max_error_number or the max_filter_ratio is more than the value set`, `Statistic` shows `"loadedRows":1` and `"errorRows":1`, and `Progress` is `{"0":"1"}`. The error log names the row: `Reason: column(order_amount) value is incorrect while strict mode is true, src value is not-a-number. src line [930002 2 CREATED not-a-number]`.

`ext_kafka_practice` holds 930001. `max_error_number` = 0 pauses the job at the first bad row, but it does not undo the rows of the same task that fit, and the job's progress has already moved past the bad message: `RESUME` would continue after it, so a corrected order has to be sent again. Compare Stream Load in Lab 5, where `strict_mode` with `max_filter_ratio` = 0 rejects the whole batch.

## 6. Stop the job and keep the results

`STOP` ends a job for good: unlike a paused job, a stopped one cannot be resumed, and a new job would start from its own default offsets. The cell stops this run's job, deletes only this run's topic, and closes the connection; the rows stay in Doris.

In [ ]:
lab.execute(f"STOP ROUTINE LOAD FOR {job_name}")
print(kafka("kafka-topics.sh", "--delete", "--topic", topic))
lab.close()

**Expected result**

No error. `SHOW ROUTINE LOAD` no longer lists the job; `SHOW ALL ROUTINE LOAD` still shows it as STOPPED.

## You are done when

- you can explain why the third order stayed in Kafka while the job was paused, and where the job kept its position;
- you can read `State`, `Progress`, `Lag`, and `Statistic` of a Routine Load job;
- you can say what `max_error_number` = 0 does to a job, to the rows of the same task, and to its progress.

To stop Kafka and keep its data, run this in the course directory:

```bash
docker compose --project-name doris-warehousing-streaming --file environments/streaming/compose.yml --profile kafka stop
```

Before you go back to the main Labs, run Lab 1's **Start and Connect** cell: it recreates the Doris container without the 6 GiB limit and keeps its data.

**If something went wrong:**

- The first cell refused to start: give Docker the memory and CPUs that it names.
- `MEM_LIMIT_EXCEEDED` or `MEM_ALLOC_FAILED`: Lab 1's startup cell may have removed Doris's 6 GiB limit. Finish the other notebooks and run this Lab's first cell again, which sets the limit back; with it, `docker inspect doris-warehousing-course-doris-1 --format '{{.HostConfig.Memory}}'` shows 6442450944.
- A wait stopped because the job paused: its card shows `ReasonOfStateChanged` and `ErrorLogUrls`, and the statement to run for the rest. Doris reaches Kafka at `course-stream-kafka:9092` on the course's Docker network.
- The Lab was interrupted: run it again from the first cell. It stops the jobs that an earlier run of this Lab left, `course_orders_<12 hexadecimal digits>` and `course_practice_<12 hexadecimal digits>`, and refuses to touch any other Routine Load job.
- Kafka's log: `docker compose --project-name doris-warehousing-streaming --file environments/streaming/compose.yml logs --tail 100 kafka`.
Start and Connect also restores the sandbox's existing lake, dashboard, and backup network connections. If you built those services earlier, check that the lake query and saved Superset charts still load; their data and grants are retained.


## Next

Take the [continuous ingestion quiz](quiz05_continuous_ingestion.ipynb) if you have not, or return to the [Module 5 lesson](course.md), or try [Optional Lab 5B: MySQL → Flink CDC → Doris](optional05_flink_mysql_cdc.ipynb). Module 6 starts with the [Module 6 lesson](../module06-data-quality/course.md).